# 03 · Videoaktionen mit T-DEED

Auf fünf automatisch ausgewählten Clips von je zwölf Sekunden werden bestehende T-DEED-Gewichte angewendet. Ziel ist zunächst, Aktionsvorhersagen für die Kontaktdiagnose in Notebook 04 zu erhalten. Scores sind unkalibriert; Team und Spieler bleiben unbekannt. Benötigt wird die separate Video-Umgebung aus `artifacts/video_event_spotting/README.md`.

[← Notebook 02](../02_spatial_hybrid_v1/02_spatial_hybrid_v1.ipynb) · [Notebook 04 →](../04_kontaktdiagnose/04_kontaktdiagnose.ipynb)

In [ ]:
from pathlib import Path
from hashlib import sha256
from fractions import Fraction
from collections import deque, defaultdict
from types import ModuleType, SimpleNamespace
from time import perf_counter
import ast
import base64
import html
import copy
import gc
import importlib.metadata
import json
import resource
import shutil
import subprocess
import sys
import xml.etree.ElementTree as ET
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import torch
from torch.utils.data import IterableDataset
from IPython.display import HTML, display

## 1. Dateien und Videozeitachse

Video und Tracking verwenden dieselbe 25-FPS-Zeitachse. Die Frameanzahl und Paketzeitstempel werden aus dem lokalen Video gelesen.

In [ ]:
PROJECT = next(p for p in (Path.cwd(), *Path.cwd().parents)
               if (p / "tactical/BAR-ATH_tactical.mp4").is_file())

VIDEO = PROJECT / "tactical/BAR-ATH_tactical.mp4"

RAW = PROJECT / "tactical/BAR-ATH_tactical_atd.txt"

META = PROJECT / "tactical/BAR-ATH_tactical_atd.xml"

HOMOGRAPHY = PROJECT / "tactical/BAR-ATH_tactical_homography.csv"

TAGS = PROJECT / "tactical/BAR-ATH_tactical_smart_tagging.xml"

ARCHIVE = PROJECT / "reference/mtah2027_official_3de6b60.zip"

OUT = PROJECT / "output/video_event_spotting_pilot"

for name in ("diagnostics", "figures", "manifests"):
    (OUT / name).mkdir(parents=True, exist_ok=True)

CLIPS = OUT / "diagnostics/clips"

CLIPS.mkdir(exist_ok=True)

FPS, N_FRAMES, RANDOM_STATE = 25, 199319, 42

HALVES = [(11185, 90651), (110808, 190197)]

FFPROBE, FFMPEG = shutil.which("ffprobe"), shutil.which("ffmpeg")

if FFPROBE is None or FFMPEG is None:
    raise RuntimeError("Vorhandene ffprobe/ffmpeg-Installation erforderlich; keine automatische Installation.")

def probe(path):
    return json.loads(subprocess.check_output([FFPROBE, "-v", "error", "-show_streams", "-show_format", "-of", "json", str(path)]))

In [ ]:
metadata = probe(VIDEO)
video_stream = next(s for s in metadata["streams"] if s["codec_type"] == "video")
(OUT / "diagnostics/video_ffprobe_metadata.json").write_text(json.dumps(metadata, indent=2))
packet_result = subprocess.run([FFPROBE, "-v", "error", "-select_streams", "v:0", "-show_packets",
                                "-show_entries", "packet=pts,dts,duration", "-of", "json", str(VIDEO)],
                               capture_output=True, text=True, check=True, timeout=240)
packets = pd.DataFrame(json.loads(packet_result.stdout)["packets"])
packets = packets.astype("int64")
time_base = Fraction(video_stream["time_base"])
if Fraction(video_stream["avg_frame_rate"]) != FPS or len(packets) != N_FRAMES:
    raise ValueError("Videozeitachse passt nicht zum vorbereiteten Tracking.")
video_time = packets.pts.to_numpy() * float(time_base)

print("Videoframes:", len(packets), "| Auflösung:", video_stream["width"], "×", video_stream["height"])

## 2. Fünf Pilotclips auswählen

Je ein automatischer Tag für Aufbau, offensive und defensive Umschaltphase, Einwurf und Abstoß bestimmt einen zwölfsekündigen Clip. Gewählt wird der erste passende, nicht überlappende Tag nach dem öffentlichen Fenster innerhalb einer Halbzeit. Die Kontextnamen bestätigen keine einzelne Aktion.

In [ ]:
tag_rows = []
for node in ET.parse(TAGS).getroot().findall(".//instance"):
    labels = {item.findtext("group"): item.findtext("text") for item in node.findall("label")}
    tag_rows.append({"tag_id": node.findtext("ID"), "start_s": float(node.findtext("start")),
                     "end_s": float(node.findtext("end")), "code": node.findtext("code"), **labels})
tag_table = pd.DataFrame(tag_rows).sort_values(["start_s", "tag_id"])
contexts = [("build_up", "BUILD UP", None), ("attacking_transition", "ATTACKING TRANSITION", None),
            ("defending_transition", "DEFENSIVE TRANSITION", None),
            ("throw_in", "SET PIECES", "Throw-in"), ("goal_kick", "SET PIECES", "Goal Kick")]
scene_rows = []
for name, code, kind in contexts:
    candidates = tag_table.loc[tag_table["code"].str.upper().eq(code)]
    if kind is not None:
        candidates = candidates.loc[candidates["Type"].str.casefold().eq(kind.casefold())]
    for row in candidates.to_dict("records"):
        center = round((row["start_s"] + row["end_s"]) * FPS / 2)
        first, stop = center - 150, center + 150
        if (first > 18684 and any(a <= first < stop <= b + 1 for a, b in HALVES)
                and not any(max(first, previous["first_frame"]) < min(stop, previous["stop_frame_exclusive"]) for previous in scene_rows)):
            scene_rows.append({"scene_id": name, "tag_id": row["tag_id"], "context_code": code,
                               "context_type": kind, "first_frame": first, "stop_frame_exclusive": stop,
                               "center_frame": center, "source_start_s": float(video_time[first]),
                               "duration_s": (stop - first) / FPS, "selection": "automatic tag; not event GT"})
            break
    else:
        raise ValueError(f"Kein geeigneter vorab definierter Kontext: {name}")
scenes = pd.DataFrame(scene_rows)
scenes.to_csv(OUT / "diagnostics/pilot_scene_selection.csv", index=False)
display(scenes[["scene_id", "first_frame", "stop_frame_exclusive", "source_start_s", "duration_s"]])

## 3. Tracking und Kameraprojektion

ATD-Zeilen sind 1-basiert, Videoframes 0-basiert. Für die Szenenmittelpunkte werden Trackingpunkte mit der zugehörigen Homographie ins lokale Videobild projiziert. Die Bilder dienen der räumlichen Sichtprüfung.

In [ ]:
sample_frames = set(scenes.center_frame.astype(int))
loaded_ids, ball_visible, visible_slots = [], [], []
sample_tracking = {}
with RAW.open() as stream:
    for frame, line in enumerate(stream):
        raw_id, players_raw, ball_raw = line.strip().split(":")
        players = np.fromstring(players_raw.replace(";", ","), sep=",").reshape(-1, 3)[:, :2]
        ball = np.fromstring(ball_raw, sep=",")[:2]
        loaded_ids.append(int(raw_id))
        ball_visible.append(bool(np.isfinite(ball).all()))
        visible_slots.append(int(np.isfinite(players).all(axis=1).sum()))
        if frame in sample_frames:
            sample_tracking[frame] = (players, ball)
loaded_ids = np.asarray(loaded_ids)
ball_visible, visible_slots = np.asarray(ball_visible), np.asarray(visible_slots)
homography = pd.read_csv(HOMOGRAPHY).set_index("frame")

print("Trackingzeilen:", len(loaded_ids), "| Ballbeobachtung:", round(float(ball_visible.mean()), 3))

In [ ]:
meta_root = ET.parse(META).getroot()
team_nodes = meta_root.findall("./Metadata/Teams/Team")
team_side = {node.get("id"): side for node, side in zip(team_nodes, ("H", "A"))}
channel_to_track = {node.get("id"): node.get("playerId") for node in meta_root.findall("./Metadata/PlayerChannels/PlayerChannel")}
spec = meta_root.find("./DataFormatSpecifications/DataFormatSpecification")
track_ids = [channel_to_track[node.find("PlayerChannelRef").get("playerChannelId")]
             for node in spec.findall("./SplitRegister/SplitRegister")]
player_team = {node.get("id"): team_side[node.get("teamId")] for node in meta_root.findall("./Metadata/Players/Player")}
slot_sides = np.array([player_team[track_id] for track_id in track_ids])
cap = cv2.VideoCapture(str(VIDEO))
fig, axes = plt.subplots(3, 2, figsize=(15, 13), layout="constrained")
image_checks = []
for ax, scene in zip(axes.flat, scenes.to_dict("records")):
    frame = scene["center_frame"]
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame)
    ok, image_bgr = cap.read()
    if not ok:
        raise RuntimeError(f"Frame {frame} nicht dekodierbar")
    image = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    ax.imshow(image)
    hrow = homography.loc[frame]
    H = hrow[[f"h{i}{j}" for i in range(3) for j in range(3)]].to_numpy(float).reshape(3, 3)
    players, ball = sample_tracking[frame]
    points = np.vstack([players, ball])
    projected = np.c_[points, np.ones(len(points))] @ H.T
    projected = projected[:, :2] / projected[:, 2, None]
    for side, color in (("H", "#2F8CFF"), ("A", "#F34C61")):
        selected = slot_sides == side
        ax.scatter(projected[:-1][selected, 0], projected[:-1][selected, 1], s=22, facecolors="none", edgecolors=color, linewidths=1)
    ax.scatter(projected[-1, 0], projected[-1, 1], s=55, color="yellow", marker="x", linewidths=2)
    ax.set(title=f"{scene['scene_id']} · Frame {frame} · PTS {video_time[frame]:.2f} s", xlim=(0, video_stream["width"]), ylim=(video_stream["height"], 0))
    ax.axis("off")
    image_checks.append({"frame": frame, "decoder_next_frame": cap.get(cv2.CAP_PROP_POS_FRAMES),
                         "decoder_timestamp_s": cap.get(cv2.CAP_PROP_POS_MSEC) / 1000,
                         "expected_pts_s": video_time[frame], "homography_valid": bool(hrow["valid"])})
for ax in axes.flat[len(scenes):]:
    ax.axis("off")
cap.release()
fig.suptitle("Taktische Perspektive · projizierte Trackingpunkte · automatische Kontexte, keine Event-GT")
fig.savefig(OUT / "figures/tracking_projection_samples.png", dpi=130, bbox_inches="tight")
plt.show(); plt.close(fig)
pd.DataFrame(image_checks).to_csv(OUT / "diagnostics/decoded_sample_alignment.csv", index=False)

## 4. Clips lokal vorbereiten

Die fünf Ausschnitte werden mit 796 × 448 Pixeln gespeichert. Clipstart und globale Frameadresse bleiben in der Clip-Tabelle erhalten.

In [ ]:
clip_rows = []
for scene in scenes.to_dict("records"):
    path = CLIPS / f"{scene['scene_id']}_{scene['first_frame']}_{scene['stop_frame_exclusive'] - 1}.mp4"
    subprocess.run([FFMPEG, "-hide_banner", "-loglevel", "error", "-y", "-ss", f"{scene['source_start_s']:.8f}",
                    "-i", str(VIDEO), "-map", "0:v:0", "-frames:v", str(scene["stop_frame_exclusive"] - scene["first_frame"]),
                    "-vf", "scale=796:448", "-c:v", "libx264", "-preset", "veryfast", "-crf", "23", "-an", str(path)], check=True)
    clip_stream = next(s for s in probe(path)["streams"] if s["codec_type"] == "video")
    clip_rows.append({**scene, "source_clip": str(path.relative_to(PROJECT)), "clip_frames": 300,
                      "clip_start_pts_s": float(clip_stream["start_time"]), "clip_bytes": path.stat().st_size,
                      "mapping": "tracking_frame = first_frame + clip_frame; timestamp uses original PTS"})
clip_manifest = pd.DataFrame(clip_rows)
clip_manifest.to_csv(OUT / "diagnostics/clip_manifest.csv", index=False)
print("Vorbereitete Sekunden:", clip_manifest.duration_s.sum(), "| Byte:", clip_manifest.clip_bytes.sum())

## 5. T-DEED und Gewichte laden

Verwendet wird `SoccerNetBall_challenge1`, Revision `d953b4de8a2ad941c565e1683c5b07255cbe644e`. Die vorhandenen Fremdquellen werden mit eigenem Modulnamensraum geladen; Backbone-Autodownload und CUDA-fester Kopf werden für die lokale CPU-Ausführung angepasst. Der vollständige Checkpoint wird strikt geladen. Code-, Gewichts- und Datensatzrechte sind getrennt in den Pilotunterlagen dokumentiert.

In [ ]:
REVISION = "d953b4de8a2ad941c565e1683c5b07255cbe644e"

ART = PROJECT / "artifacts/video_event_spotting"

SOURCE = ART / "source" / REVISION

CHECKPOINT = ART / "weights/tdeed_soccernetball_challenge1_checkpoint_best.pt"

WEIGHT_SHA = "f7b5cee134a2cfffc5b76922bbf528fe24cb6cc35c550dba23b9eb1bd4ff3d35"

THRESHOLD, STRIDE, NMS_WINDOW = 0.2, 2, 12

required_versions = {"torch": "2.3.1", "torchvision": "0.18.1", "timm": "1.0.3", "numpy": "1.26.4",
                     "pandas": "2.3.3", "opencv-python": "4.10.0.84"}

versions = {name: importlib.metadata.version(name) for name in required_versions}

if versions != required_versions:
    raise RuntimeError(f"Bitte isolierte Video-Umgebung verwenden: {versions}")

if sha256(CHECKPOINT.read_bytes()).hexdigest() != WEIGHT_SHA:
    raise ValueError("Der Checkpoint entspricht nicht der dokumentierten T-DEED-Version.")

torch.manual_seed(RANDOM_STATE)

np.random.seed(RANDOM_STATE)

torch.set_num_threads(4)

patches = []

namespace = "_mtah_tdeed"

for package in (namespace, namespace + ".impl"):
    module = ModuleType(package)
    module.__path__ = []
    sys.modules[package] = module

for name, relative in [("modules", "model/modules.py.txt"), ("impl.gsm", "model/impl/gsm.py.txt"),
                       ("impl.gsf", "model/impl/gsf.py.txt"), ("shift", "model/shift.py.txt"),
                       ("model", "model/model.py.txt")]:
    path = SOURCE / relative
    text = path.read_text().replace("from model.", f"from {namespace}.")
    if name == "model":
        for old, new in [("pretrained=True", "pretrained=False"),
                         ("self._pred_fine.cuda()", "self._pred_fine.to(self.temp_enc.device)")]:
            text = text.replace(old, new)
            patches.append({"file": relative, "old": old, "new": new})
    module = ModuleType(f"{namespace}.{name}")
    module.__file__ = str(path)
    sys.modules[module.__name__] = module
    exec(compile(text, str(path), "exec"), module.__dict__)

def original_nodes(relative, names, scope):
    path = SOURCE / relative
    tree = ast.parse(path.read_text())
    nodes = [node for node in tree.body if isinstance(node, (ast.FunctionDef, ast.ClassDef)) and node.name in names]
    exec(compile(ast.Module(body=nodes, type_ignores=[]), str(path), "exec"), scope)

foreign = {"torch": torch, "np": np, "cv2": cv2, "deque": deque, "IterableDataset": IterableDataset,
           "DEFAULT_PAD_LEN": 5, "copy": copy, "defaultdict": defaultdict}

original_nodes("dataset/frame.py.txt", ["ActionSpotInferenceDataset"], foreign)

original_nodes("util/eval.py.txt", ["process_frame_predictions_inference", "soft_non_maximum_supression"], foreign)

ActionSpotInferenceDataset = foreign["ActionSpotInferenceDataset"]

config = json.loads((SOURCE / "config/SoccerNetBall/SoccerNetBall_challenge1.json.txt").read_text())

args = SimpleNamespace(**{**config, "batch_size": 1, "crop_dim": None, "num_workers": 0})

classes = {name: index for index, name in enumerate((SOURCE / "data/soccernetball/class.txt.txt").read_text().splitlines(), 1)}

runtime = {"python": sys.version, "prefix": sys.prefix, "packages": versions, "threads": torch.get_num_threads(),
           "source_revision": REVISION, "weight_sha256": WEIGHT_SHA,
           "execution_patches": patches, "namespace_patch": f"from model.* -> from {namespace}.*",
           "settings": {"batch_size": 1, "fp32": True, "amp": False, "tta": False, "threshold": THRESHOLD,
                        "stride": STRIDE, "clip_len": args.clip_len, "overlap_len": 75, "pad_len": 5,
                        "soft_nms_window_sampled_frames": NMS_WINDOW}}

(OUT / "manifests/inference_runtime.json").write_text(json.dumps(runtime, indent=2))


display(pd.Series(versions, name="Version").to_frame())
print("Native Klassen:", list(classes))

In [ ]:
start_load = perf_counter()
model = sys.modules[f"{namespace}.model"].TDEEDModel(device="cpu", args=args)
model._model.update_pred_head([13, 18])
model._num_classes = 31
state = torch.load(CHECKPOINT, map_location="cpu", weights_only=True)
load_result = model._model.load_state_dict(state, strict=True)
parameter_count = sum(p.numel() for p in model._model.parameters())
checkpoint_load = {"weights_only": True, "strict": True, "tensor_count": len(state),
                   "missing_keys": list(load_result.missing_keys), "unexpected_keys": list(load_result.unexpected_keys),
                   "parameters": parameter_count, "load_seconds": perf_counter() - start_load}
del state
gc.collect()
(OUT / "manifests/checkpoint_load.json").write_text(json.dumps(checkpoint_load, indent=2))

print("Geladene Modellparameter:", parameter_count)

## 6. Inferenz auf allen fünf Clips

CPU, FP32, Batch 1; Fensterlänge 100, Stride 2, Überlappung 75, Padding 5. Originalschwelle 0,2 und Soft-NMS-Radius 12 bleiben unverändert. Gespeichert werden die Vorhersagen sowie Scores, Unterstützung und Fensterabdeckung. Unvollständige Endfenster bleiben wie im Original unberücksichtigt; die letzten 0,4 s eines Clips sind nicht abgedeckt.

In [ ]:
PRED = OUT / "predictions"
PRED.mkdir(exist_ok=True)
SCORES = OUT / "diagnostics/model_scores"
SCORES.mkdir(exist_ok=True)
clip_manifest = pd.read_csv(OUT / "diagnostics/clip_manifest.csv")
columns = ["video_timestamp_s", "tracking_frame", "event_type", "event_confidence", "team_side", "player_track_id",
           "model_name", "model_version", "source_clip"]
model_version = f"{REVISION};sha256:{WEIGHT_SHA}"


def infer_clip(scene):
    dataset = ActionSpotInferenceDataset(str(PROJECT / scene["source_clip"]), clip_len=100, overlap_len=75,
                                         stride=STRIDE, pad_len=5, dataset="soccernetball", size=(796, 448))
    scores = np.zeros((dataset._video_len // STRIDE, 13), np.float32)
    support = np.zeros(len(scores), np.int32)
    coverage = np.zeros(len(scores), np.int32)
    starts, window_seconds = [], []
    started = perf_counter()
    for frames, start in dataset:
        before = perf_counter()
        _, predicted = model.predict(frames, use_amp=False, augment_inference=False)
        window_seconds.append(perf_counter() - before)
        starts.append(int(start))
        values = predicted[0]
        if start < 0:
            values, start = values[-start:], 0
        end = min(start + len(values), len(scores))
        values = values[:end - start]
        scores[start:end] += values
        support[start:end] += values.sum(axis=1) != 0
        coverage[start:end] += 1
        print(scene["scene_id"], "Fenster", len(starts), f"{window_seconds[-1]:.1f} s", flush=True)
        del frames, predicted, values
        gc.collect()
    np.savez_compressed(SCORES / f"{scene['scene_id']}.npz", summed_scores=scores, support=support,
                        window_coverage=coverage, sampled_local_frames=np.arange(len(scores)) * STRIDE,
                        window_starts=starts)
    _, high_recall, averaged = foreign["process_frame_predictions_inference"](
        "soccernetball", classes, scores.copy(), support.copy(), high_recall_score_threshold=THRESHOLD)
    events = foreign["soft_non_maximum_supression"]([{"events": high_recall}], window=NMS_WINDOW, threshold=THRESHOLD)[0]["events"]
    rows = []
    for event in events:
        local_frame = int(event["frame"]) * STRIDE
        global_frame = int(scene["first_frame"]) + local_frame
        rows.append(dict(zip(columns, [global_frame / FPS, global_frame, event["label"], event["score"],
                                       "unknown", "", "T-DEED SoccerNetBall_challenge1", model_version, scene["source_clip"]])))
    table = pd.DataFrame(rows, columns=columns)
    table.to_csv(PRED / f"{scene['scene_id']}_events.csv", index=False)
    elapsed = perf_counter() - started
    summary = {"scene_id": scene["scene_id"], "device": model.device, "clip_seconds": float(scene["duration_s"]),
               "wall_seconds": elapsed, "inference_window_seconds": window_seconds, "window_starts": starts,
               "sampled_positions": len(scores), "geometrically_covered_positions": int((coverage > 0).sum()),
               "supported_positions": int((support > 0).sum()), "events": len(table),
               "events_per_minute_clip_duration": len(table) * 60 / scene["duration_s"],
               "process_peak_rss_mib": resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 2**20}
    (OUT / f"diagnostics/{scene['scene_id']}_inference.json").write_text(json.dumps(summary, indent=2))
    return table, summary


pilot_tables, pilot_runtime = [], []
for scene in clip_manifest.to_dict("records"):
    table, timing = infer_clip(scene)
    pilot_tables.append(table)
    pilot_runtime.append(timing)
video_events = pd.concat(pilot_tables, ignore_index=True).sort_values(["tracking_frame", "event_type"]).reset_index(drop=True)
video_events.to_csv(PRED / "video_events_pilot.csv", index=False)
runtime_table = pd.DataFrame(pilot_runtime)
runtime_table.to_json(OUT / "diagnostics/clip_inference_summary.json", orient="records", indent=2)
display(runtime_table[["scene_id", "wall_seconds", "supported_positions", "events"]])

## 7. Vorhersagen im zeitlichen Kontext

Die Klassen- und Scoreverteilung beschreibt die Ausgabe. v1-Grenzen dienen als bestehender Modellkontext, nicht als Event-Ground-Truth. Das Ausgaberaster beträgt 80 ms; semantische Aktions- und Zeitgenauigkeit sind nicht unabhängig validiert.

In [ ]:
class_counts = video_events.event_type.value_counts().rename_axis("event_type").rename("events").to_frame()
class_counts.to_csv(OUT / "diagnostics/predicted_class_counts.csv")
display(class_counts)
print(f"{len(video_events)} Vorhersagen / 60 s Clipmaterial; keine gemessene Präzision oder Recall.")
display(video_events.event_confidence.describe().round(3).to_frame())

v1 = pd.read_csv(PROJECT / "artifacts/spatial_hybrid_v1/references/spatial_hybrid_v1.csv")
labels = v1.label.to_numpy()
switches = np.flatnonzero((labels[1:] != labels[:-1]) & np.isin(labels[1:], ["H", "A"]) & np.isin(labels[:-1], ["H", "A"])) + 1
context_rows = []
fig, axes = plt.subplots(5, 1, figsize=(13, 13), constrained_layout=True)
class_index = {name: i for i, name in enumerate(classes)}
for ax, scene in zip(axes, clip_manifest.to_dict("records")):
    subset = video_events[video_events.source_clip == scene["source_clip"]]
    local_switches = switches[(switches >= scene["first_frame"]) & (switches < scene["stop_frame_exclusive"])]
    for frame in local_switches:
        ax.axvline((frame - scene["first_frame"]) / FPS, color="gray", ls="--", alpha=0.5)
    for row in subset.itertuples():
        ax.scatter((row.tracking_frame - scene["first_frame"]) / FPS, class_index[row.event_type],
                   s=30 + row.event_confidence * 130, color="tab:blue")
        distance = float(np.min(np.abs(local_switches - row.tracking_frame)) / FPS) if len(local_switches) else None
        context_rows.append({"tracking_frame": row.tracking_frame, "event_type": row.event_type,
                             "scene_id": scene["scene_id"], "nearest_v1_switch_distance_s": distance,
                             "meaning": "descriptive distance; not event or switch correctness"})
    coverage = np.load(SCORES / f"{scene['scene_id']}.npz")["window_coverage"]
    tail_start_s = np.flatnonzero(coverage > 0)[-1] * STRIDE / FPS + STRIDE / FPS
    ax.axvspan(tail_start_s, 12, color="lightgray", alpha=0.6)
    ax.set(title=f"{scene['scene_id']}: {len(subset)} Event-Vorhersagen; {len(local_switches)} v1-H/A-Grenzen (gestrichelt)",
           xlim=(0, 12), ylim=(-0.5, len(classes) - 0.5), yticks=range(len(classes)), yticklabels=list(classes), xlabel="Lokale Clipzeit [s]")
    ax.tick_params(axis="y", labelsize=7)
fig.savefig(OUT / "figures/event_timeline_v1_context.png", dpi=150)
plt.show()
pd.DataFrame(context_rows).to_csv(OUT / "diagnostics/event_v1_switch_context.csv", index=False)
print("Beobachtung: Verteilung und Kontext sind deskriptiv; semantische Korrektheit und Event-Zeitfehler bleiben unbekannt.")

## 8. Lokale Videoansicht

Die Montage zeigt die gespeicherten Vorhersagen mit Klasse, Score und globalem Frame. Die Einblendung von ±0,4 s dient der Lesbarkeit. Bilder und Player bleiben lokal; Notebook-Ausgaben vor einer Veröffentlichung entfernen.

In [ ]:
preview_project = next(p for p in (Path.cwd(), *Path.cwd().parents)
                       if (p / "tactical/BAR-ATH_tactical.mp4").is_file())
preview_out = preview_project / "output/video_event_spotting_pilot"
view_dir = preview_out / "diagnostics/annotated"
view_dir.mkdir(exist_ok=True)
view_events = pd.read_csv(preview_out / "predictions/video_events_pilot.csv", keep_default_na=False)
view_clips = pd.read_csv(preview_out / "diagnostics/clip_manifest.csv")
view_path = view_dir / "video_events_pilot_annotated.mp4"
view_fps, width, height, panel_height = 25, 796, 448, 164
view_colors = {"PASS": (90, 220, 130), "DRIVE": (255, 180, 70), "HIGH PASS": (100, 200, 255)}
view_index, rendered_frames = [], 0
encoder = subprocess.Popen([shutil.which("ffmpeg"), "-hide_banner", "-loglevel", "error", "-y",
                            "-f", "rawvideo", "-pixel_format", "bgr24", "-video_size", f"{width}x{height + panel_height}",
                            "-framerate", str(view_fps), "-i", "pipe:0", "-an", "-c:v", "libx264",
                            "-preset", "fast", "-crf", "20", "-pix_fmt", "yuv420p", "-movflags", "+faststart", str(view_path)],
                           stdin=subprocess.PIPE)
try:
    for scene in view_clips.to_dict("records"):
        clip_events = view_events[view_events.source_clip == scene["source_clip"]].copy()
        clip_events["local_frame"] = clip_events.tracking_frame - scene["first_frame"]
        montage_start_frame = rendered_frames
        for row in clip_events.to_dict("records"):
            view_index.append({**row, "scene_id": scene["scene_id"], "clip_time_s": row["local_frame"] / view_fps,
                               "montage_time_s": (montage_start_frame + row["local_frame"]) / view_fps,
                               "playback_start_s": (montage_start_frame + max(0, row["local_frame"] - 15)) / view_fps})
        coverage = np.load(preview_out / f"diagnostics/model_scores/{scene['scene_id']}.npz")["window_coverage"]
        stream = cv2.VideoCapture(str(preview_project / scene["source_clip"]))
        try:
            for local_frame in range(int(scene["clip_frames"])):
                ok, image = stream.read()
                if not ok:
                    raise RuntimeError(f"Clip unerwartet zu Ende: {scene['scene_id']} / {local_frame}")
                frame = np.zeros((height + panel_height, width, 3), np.uint8)
                frame[:height] = image
                frame[height:] = (22, 22, 22)
                global_frame = int(scene["first_frame"]) + local_frame
                cv2.putText(frame, f"{scene['scene_id']} | Quelle {global_frame / view_fps:.2f} s | Frame {global_frame}",
                            (12, height + 25), cv2.FONT_HERSHEY_SIMPLEX, 0.52, (245, 245, 245), 1, cv2.LINE_AA)
                active = clip_events[(clip_events.local_frame - local_frame).abs() <= 10]
                caption = " | ".join(f"{r.event_type}: {r.event_confidence:.2f} @ Fr {r.tracking_frame}" for r in active.itertuples())
                cv2.putText(frame, "VORHERSAGE: " + (caption or "kein Event im Anzeigefenster"),
                            (12, height + 52), cv2.FONT_HERSHEY_SIMPLEX, 0.47, (130, 235, 190), 1, cv2.LINE_AA)
                cv2.putText(frame, "Team unbekannt | Score unkalibriert | Anzeige +/-0.40 s, keine Aktionsdauer",
                            (12, height + 78), cv2.FONT_HERSHEY_SIMPLEX, 0.43, (205, 205, 205), 1, cv2.LINE_AA)
                if coverage[local_frame // 2] == 0:
                    cv2.putText(frame, "OHNE MODELLABDECKUNG", (12, height + 103), cv2.FONT_HERSHEY_SIMPLEX, 0.48, (180, 180, 255), 1)
                line_y, left, span = height + 139, 15, width - 30
                cv2.line(frame, (left, line_y), (left + span, line_y), (115, 115, 115), 2)
                cv2.rectangle(frame, (int(left + span * 290 / 300), line_y - 12), (left + span, line_y + 12), (65, 65, 65), -1)
                for row in clip_events.itertuples():
                    x = int(left + span * row.local_frame / 300)
                    cv2.line(frame, (x, line_y - 10), (x, line_y + 10), view_colors[row.event_type], 2)
                cv2.circle(frame, (int(left + span * local_frame / 300), line_y), 5, (255, 255, 255), -1)
                encoder.stdin.write(frame.tobytes())
                rendered_frames += 1
        finally:
            stream.release()
finally:
    encoder.stdin.close()
    returncode = encoder.wait()
if returncode:
    raise RuntimeError(f"ffmpeg-Export fehlgeschlagen: {returncode}")
index_table = pd.DataFrame(view_index)
index_table.to_csv(view_dir / "event_video_index.csv", index=False)

# Lokaler Player: dieselben gespeicherten Vorhersagen, keine weiteren Bibliotheken oder Netzwerkzugriffe.
buttons = "\n".join(f'<button data-start="{r.playback_start_s:.2f}">{html.escape(r.scene_id)} · {r.clip_time_s:.2f}s · '
                    f'{html.escape(r.event_type)} · Score {r.event_confidence:.2f} · Frame {r.tracking_frame}</button>'
                    for r in index_table.itertuples())
player = """<!doctype html><html lang="de"><meta charset="utf-8"><title>T-DEED: Pilot im Video</title>
<style>body{font:15px system-ui;background:#15191e;color:#eee;margin:16px}video{width:100%;max-height:65vh;background:#000}
h2{font-size:20px}p{color:#c5cbd2}#events{max-height:260px;overflow:auto;display:grid;gap:6px;margin-top:12px}
button{font:inherit;text-align:left;padding:9px;background:#26303a;color:#eee;border:1px solid #46505b;border-radius:5px;cursor:pointer}
button:hover{background:#3b4d5e}</style><h2>T-DEED · 47 unvalidierte Aktionsvorhersagen</h2>
<p>Event anklicken: Wiedergabe startet 0,60 s vorher im jeweiligen Clip. Leertaste bzw. Videosteuerung zum Pausieren.
Die Einblendung von ±0,40 s dient der Lesbarkeit. Keine Team-/Spielerzuordnung.</p>
<video id="video" controls playsinline preload="metadata" src="__VIDEO__"></video><div id="events">__BUTTONS__</div>
<script>const video=document.getElementById('video');document.querySelectorAll('button').forEach(button=>{
button.onclick=()=>{video.currentTime=Number(button.dataset.start);video.play();};});</script></html>"""
player = player.replace("__BUTTONS__", buttons)
player_path = view_dir / "event_spotting_player.html"
player_path.write_text(player.replace("__VIDEO__", view_path.name))
embedded_player = player.replace("__VIDEO__", "data:video/mp4;base64," + base64.b64encode(view_path.read_bytes()).decode())
display(HTML('<iframe sandbox="allow-scripts" style="width:100%;height:940px;border:0" srcdoc="' + html.escape(embedded_player, quote=True) + '"></iframe>'))
display(index_table[["scene_id", "clip_time_s", "montage_time_s", "tracking_frame", "event_type", "event_confidence"]])
print("Video:", view_path)
print("Anklickbarer lokaler Player:", player_path)

## Ergebnis

Der bisherige Pilot liefert 47 Vorhersagen: 25 PASS, 20 DRIVE und 2 HIGH PASS. Eine gemessene Präzision oder ein Besitzgewinn folgt daraus nicht. Notebook 04 vergleicht die Zeitpunkte mit den ursprünglichen Trackingkontakten.